In [1]:
########## import some useful package ##########

import time
t1 = time.time()

import uproot
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import ripser
import persim

In [2]:
########## read the data ##########

processes = ["jjBG_ptcut", "ttBG_ptcut", "bbBG_ptcut", "wwBG", "wwvvBG", "zzBG", "zzvvBG", "hzvvBG"]
event_type = [str(i)+"_500K" for i in processes]

BR_hbb = 0.5824
Xsection = [1.08143e-1, 3.539e-2, 1.86879e-2, 2.200885e+1, 7.053723e+1, 1.3355, 3.983523e+1, 7.03508147*BR_hbb]     ### unit: fb
Luminosity = 1000   ### unit: fb^-1
simulation_num = 500000

features = ["VLCjetR10N2", "VLCjetR10N2.PT", "VLCjetR10N2.Mass", "VLCjetR10N2.Constituents", "EFlowTrack.fUniqueID", "EFlowTrack.PT", "EFlowTrack.Eta", "EFlowTrack.Phi", "EFlowTrack.PID", "EFlowPhoton.fUniqueID", "EFlowPhoton.ET", "EFlowPhoton.Eta", "EFlowPhoton.Phi", "EFlowNeutralHadron.fUniqueID", "EFlowNeutralHadron.ET", "EFlowNeutralHadron.Eta", "EFlowNeutralHadron.Phi"]

##### set data path #####

event_path = []
for type in event_type:
    event_path.append("/data/mucollider/two_boosted/10TeV/" + type + "/delphes_output.root")

##### get the data file #####

data_file =[]
for path in event_path:
    data_file.append(uproot.open(path))
    
##### read data with features #####
events = []     ### total events
for process in processes:
    tmp_events = []
    for feature in features:
        tmp_events.append(data_file[processes.index(process)]["Delphes;1"][feature].array())
    tmp_events = np.expand_dims(tmp_events, axis=-1)
    tmp_events = tmp_events.transpose((1,0,2))
    tmp_events = np.squeeze(tmp_events,axis=(2,))
    events.append(tmp_events)
    print("Time:{:^8.4f}(s)".format(time.time()-t1))
del tmp_events

/usr/local/lib/python3.8/dist-packages/awkward/array/base.py:394: VisibleDeprecationWarning: Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.
  return cls.numpy.array(value, copy=False)


Time:59.8941 (s)
Time:114.3687(s)
Time:161.5135(s)
Time:193.2560(s)
Time:227.6025(s)
Time:255.8847(s)
Time:298.0601(s)
Time:334.5032(s)


In [3]:
########## define useful function ##########

##### calculate significance #####

def significance(s,b):
    return np.sqrt(2*((s+b)*np.log(1+s/b)-s))

##### count event number #####

def count(events):
    events_num = []
    for i, process in enumerate(processes):
        events_num.append(len(events[processes.index(process)]))
        print("There are", events_num[i], process, "events. Corresponding cross section:", Xsection[processes.index(process)]*events_num[i]/simulation_num, "(fb)")
        
##### select if Fat Jet >= 2 #####
        
def Fat_Jet_selection(events):
    where1 = np.where(events[:,features.index("VLCjetR10N2")]>=2)
    return events[where1]

##### select if M_jet_leading > XX GeV #####

def mass_leading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][0]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][0]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if M_jet_subleading > XX GeV #####

def mass_subleading_selection(events, low_mass_cut, high_mass_cut):
    where1 = []
    for i in range(len(events)):
        switch=1
        if events[i][features.index("VLCjetR10N2.Mass")][1]<low_mass_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.Mass")][1]>high_mass_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_leading > XX GeV #####

def pT_leading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][0]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][0]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]

##### select if pT_J_subleading > XX GeV #####

def pT_subleading_selection(events, low_pT_cut, high_pT_cut):
    where1 = []
    for i in range(len(events)):
        switch = 1
        if events[i][features.index("VLCjetR10N2.PT")][1]<low_pT_cut:
            switch=0
        if events[i][features.index("VLCjetR10N2.PT")][1]>high_pT_cut:
            switch=0
        if switch==1:
            where1.append(i)
    return events[where1]


In [4]:
def get_particle_info(event):
    ##### set jet constituents ID #####

    leadingJ_constituents = event[features.index("VLCjetR10N2.Constituents")][0][:-1]
    subleadingJ_constituents= event[features.index("VLCjetR10N2.Constituents")][1][:-1]

    ##### construct infromation for all EFlow particles in this events and sort by PT #####

    all_ID = np.concatenate((event[features.index("EFlowTrack.fUniqueID")], event[features.index("EFlowPhoton.fUniqueID")], event[features.index("EFlowNeutralHadron.fUniqueID")]))
    all_PT = np.concatenate((event[features.index("EFlowTrack.PT")], event[features.index("EFlowPhoton.ET")], event[features.index("EFlowNeutralHadron.ET")]))
    all_Eta = np.concatenate((event[features.index("EFlowTrack.Eta")], event[features.index("EFlowPhoton.Eta")], event[features.index("EFlowNeutralHadron.Eta")]))
    all_Phi = np.concatenate((event[features.index("EFlowTrack.Phi")], event[features.index("EFlowPhoton.Phi")], event[features.index("EFlowNeutralHadron.Phi")]))
    all_PID = np.concatenate((event[features.index("EFlowTrack.PID")], np.full(len(event[features.index("EFlowPhoton.fUniqueID")]), 22), np.full(len(event[features.index("EFlowNeutralHadron.fUniqueID")]), 4)))

    #####                          categorize PID                      #####
    ##### 1:electron 2:muon 3:photon 4:neutral hadron 5:charged hadron #####

    condlist = [abs(all_PID)==11, abs(all_PID)==13, abs(all_PID)==22, abs(all_PID)==4]
    choicelist = [1, 2, 3, 4]
    all_PID = np.select(condlist, choicelist, 5)

    sort_order = np.argsort(all_PT)[::-1]   ### from large PT to low PT

    all_ID = all_ID[sort_order]
    all_PT = all_PT[sort_order]
    all_Eta = all_Eta[sort_order]
    all_Phi = all_Phi[sort_order]
    all_PID = all_PID[sort_order]

    ##### find Jet particle in EFlow particle #####
    if len(leadingJ_constituents)>20: where_leading = np.where(np.isin(all_ID, leadingJ_constituents))[0][0:20]
    else: where_leading = np.where(np.isin(all_ID, leadingJ_constituents))[0]

    if len(subleadingJ_constituents)>20: where_subleading = np.where(np.isin(all_ID, subleadingJ_constituents))[0][0:20]
    else: where_subleading = np.where(np.isin(all_ID, subleadingJ_constituents))[0]

    leadingJ_PT, subleadingJ_PT = all_PT[where_leading], all_PT[where_subleading]
    leadingJ_Eta, subleadingJ_Eta = all_Eta[where_leading], all_Eta[where_subleading]
    leadingJ_Phi, subleadingJ_Phi = all_Phi[where_leading], all_Phi[where_subleading]
    leadingJ_PID, subleadingJ_PID = all_PID[where_leading], all_PID[where_subleading]

    leadingJ_info, subleadingJ_info = np.stack((leadingJ_PT, leadingJ_Eta, leadingJ_Phi, leadingJ_PID), axis=1).ravel(), np.stack((subleadingJ_PT, subleadingJ_Eta, subleadingJ_Phi, subleadingJ_PID), axis=1).ravel()

    ##### do zero padding #####

    particle_info = np.zeros(2*20*4)
    particle_info[0:len(leadingJ_info)] = leadingJ_info
    particle_info[80:80+len(subleadingJ_info)] = subleadingJ_info
    
    return particle_info

In [5]:
########## preselection ##########

print("Before any selection:")
count(events)

##### 2 fat jet selection #####

for process in processes:
    events[processes.index(process)] = Fat_Jet_selection(events[processes.index(process)])
print("\nAfter 2 fat jet selection:")
count(events)

##### leading jet pT selection #####

leading_low_pT_cut = 200   ### GeV
leading_high_pT_cut = 800   ### GeV
for process in processes:
    events[processes.index(process)] = pT_leading_selection(events[processes.index(process)], leading_low_pT_cut, leading_high_pT_cut)
print("\nAfter leading jet pT selection:")
count(events)

##### subleading jet pT selection #####

subleading_low_pT_cut = 200   ### GeV
subleading_high_pT_cut = 600   ### GeV
for process in processes:
    events[processes.index(process)] = pT_subleading_selection(events[processes.index(process)], subleading_low_pT_cut, subleading_high_pT_cut)
print("\nAfter subleading jet pT selection:")
count(events)

##### leading jet mass selection #####

leading_low_mass_cut = 65
leading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_leading_selection(events[processes.index(process)], leading_low_mass_cut, leading_high_mass_cut)
print("\nAfter leading jet mass selection:")
count(events)

##### subleading jet mass selection #####

subleading_low_mass_cut = 65
subleading_high_mass_cut = 150
for process in processes:
    events[processes.index(process)] = mass_subleading_selection(events[processes.index(process)], subleading_low_mass_cut, subleading_high_mass_cut)
print("\nAfter subleading jet mass selection:")
count(events)

Before any selection:
There are 500000 jjBG_ptcut events. Corresponding cross section: 0.108143 (fb)
There are 500000 ttBG_ptcut events. Corresponding cross section: 0.03539 (fb)
There are 500000 bbBG_ptcut events. Corresponding cross section: 0.0186879 (fb)
There are 500000 wwBG events. Corresponding cross section: 22.00885 (fb)
There are 500000 wwvvBG events. Corresponding cross section: 70.53723 (fb)
There are 500000 zzBG events. Corresponding cross section: 1.3355 (fb)
There are 500000 zzvvBG events. Corresponding cross section: 39.83523 (fb)
There are 500000 hzvvBG events. Corresponding cross section: 4.097231448128 (fb)

After 2 fat jet selection:
There are 499967 jjBG_ptcut events. Corresponding cross section: 0.10813586256200002 (fb)
There are 499969 ttBG_ptcut events. Corresponding cross section: 0.03538780582 (fb)
There are 499963 bbBG_ptcut events. Corresponding cross section: 0.0186865170954 (fb)
There are 301339 wwBG events. Corresponding cross section: 13.264249700299999 

In [6]:
##### generate particle information #####

particle_info = []

for process in processes:
    for event in events[processes.index(process)]:
        particle_info.append(get_particle_info(event))
    print("Time:{:^8.4f}(s)".format(time.time()-t1))

Time:392.7847(s)
Time:396.9587(s)
Time:399.2865(s)
Time:399.2954(s)
Time:411.4189(s)
Time:411.4261(s)
Time:426.5982(s)
Time:441.1234(s)


In [7]:
particle_info[0]

array([207.10401917,   2.25092745,   2.59868789,   5.        ,
       150.20053101,   2.28799891,   2.5686295 ,   4.        ,
        48.67640686,   2.22830057,   2.65203047,   4.        ,
        19.06972504,   2.24115586,   2.64336562,   5.        ,
        17.46397591,   2.24436474,   2.59448242,   5.        ,
        12.15527153,   2.26053119,   2.5993557 ,   5.        ,
        12.0404911 ,   2.24783444,   2.64254689,   5.        ,
         8.07948875,   2.2660501 ,   2.62995195,   3.        ,
         5.51483011,   2.14742899,   2.58822703,   5.        ,
         4.88357258,   2.47352791,   2.87372112,   5.        ,
         2.85345888,   2.27133727,   2.65300083,   3.        ,
         2.66691518,   1.06239116,   1.26134384,   5.        ,
         2.48042393,   1.24644542,   1.44259322,   3.        ,
         2.10552049,   1.23804188,   1.47978735,   3.        ,
         1.77915311,   1.08421683,   0.85114306,   4.        ,
         1.74198472,   2.3831687 ,   2.52389479,   5.  

In [8]:
##### output test data #####

d = {'particle_info': particle_info}
df = pd.DataFrame(data=d)
df

,particle_info
0,"[207.10401916503906, 2.250927448272705, 2.5986..."
1,"[168.12562561035156, 1.8901441097259521, 1.716..."
2,"[116.70855712890625, -2.097702980041504, 2.432..."
3,"[176.18443298339844, 1.2127442359924316, 0.957..."
4,"[36.147151947021484, 1.9213073253631592, 0.095..."
...,...
239259,"[57.00393295288086, -1.200681447982788, -0.811..."
239260,"[95.35498809814453, -0.06528924405574799, 2.64..."
239261,"[65.16289520263672, 1.9384725093841553, 2.0270..."
239262,"[103.28164672851562, -0.9109786748886108, 0.19..."


In [9]:
import h5py
h5f = h5py.File('/data/mucollider/two_boosted/10TeV/data_LLfeature.h5', 'w')
for key, value in d.items():
    h5f.create_dataset(key, data=value)
    print(f"已寫入 Dataset: {key}")
h5f.close()

已寫入 Dataset: particle_info
